In [ ]:
import sys
sys.path.insert(0, "../src")
from path_config import set_path_mapping, get_path_mapping
import json
import os

In [ ]:
# path to the folder with COBIVECO mesh results
COBIVECO_OUTPUT_FOLDER = "/mnt/data-vm2/Cobiveco/input-data"
# path to the folder with TORSO-MPP results
TORSO_PIPELINE_OUTPUT_FOLDER = "/mnt/data-vm2/TORSO-MPP/data-fitted"
# path to INPUT-DATA for Cardiac-Digital-Twin framework
CDT_DATA_FOLDER = "/mnt/cardiac-data/meta_data/"
# path to the folder to save results Cardiac-Digital-Twin framework RESULTS
CDT_RESULTS_FOLDER = "/data/Cardiac-Digital-Twin/cardiac-data/meta_data/results_test/"

path_mapping_dict = {
    "cobiveco_output_path": COBIVECO_OUTPUT_FOLDER,
    "torso_output_path": TORSO_PIPELINE_OUTPUT_FOLDER,
    "data_path": CDT_DATA_FOLDER,
    "results_path": CDT_RESULTS_FOLDER
}
set_path_mapping(json.dumps(path_mapping_dict))

In [ ]:
#import mesh from cobiveco, scaling to coarse and fine resolution and saving coordinates csv
CASE_NAME = "sb3701"
coarse_resolution = '1500'
fine_resolution = '500'
cobiveco_um = 'mm'
cobiveco_output_coarse = os.path.join(path_mapping_dict["cobiveco_output_path"], f"{CASE_NAME}_{coarse_resolution+cobiveco_um}_resFibers")
cobiveco_output_fine = os.path.join(path_mapping_dict["cobiveco_output_path"], f"{CASE_NAME}_{fine_resolution+cobiveco_um}_resFibers")

In [ ]:
from cobiveco_export_functions import scale_mesh_mantaining_point_data, save_vtk_vtu_to_csv, save_vtu_arrays_to_csv, export_electrodes_to_csv, extract_ids_from_tagged_vtp
geometric_data_dir = os.path.join(path_mapping_dict["data_path"], "geometric_data")
cdt_coarse_um = 'cm'
cdt_fine_um = 'um'
subject_geom_dir = os.path.join(geometric_data_dir, CASE_NAME)
if not os.path.exists(subject_geom_dir):
    os.makedirs(subject_geom_dir)
#SCALING MESH coarse from mm to cm for CDT framework
input_filename =  f"{CASE_NAME}_{coarse_resolution+cobiveco_um}_fibers.vtu" #from mm to cm
input_path = os.path.join(cobiveco_output_coarse, input_filename)
out_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}_fibers.vtu" #from mm to cm
out_path = os.path.join(geometric_data_dir, CASE_NAME, out_filename)

scale_mesh_mantaining_point_data(input_path, out_path, scale=0.1) # scale mm to cm

input_filename = f"{CASE_NAME}_{coarse_resolution+cobiveco_um}.vtp" #from mm to cm
input_path = os.path.join(cobiveco_output_coarse, input_filename)
out_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}.vtp"
out_path = os.path.join(geometric_data_dir, CASE_NAME, out_filename)
scale_mesh_mantaining_point_data(input_path, out_path, scale=0.1)  # scale mm to cm 

#scaling mesh fine from mm to um for monodomain simulation 
input_filename = f"{CASE_NAME}_{fine_resolution+cobiveco_um}.vtp" # da mm a um 
input_path = os.path.join(cobiveco_output_fine, input_filename)
out_filename = f"{CASE_NAME}_{fine_resolution+cdt_fine_um}.vtp" # da mm a um
out_path = os.path.join(geometric_data_dir, CASE_NAME, out_filename)
scale_mesh_mantaining_point_data(input_path, out_path, scale=1000) # scale mm to um

input_filename = f"{CASE_NAME}_{fine_resolution+cobiveco_um}_fibers.vtk" # da mm a um
input_path = os.path.join(cobiveco_output_fine, input_filename)
out_filename = f"{CASE_NAME}_{fine_resolution+cdt_fine_um}_fibers.vtk" # da mm a um
out_path = os.path.join(geometric_data_dir, CASE_NAME, out_filename)
scale_mesh_mantaining_point_data(input_path, out_path, scale=1000)

coarse_folder_tag = 'coarse'+coarse_resolution+cdt_coarse_um
fine_folder_tag = 'fine'+fine_resolution+cdt_fine_um
# EXPORT CSV nodi, tetra, coordinate cobiveco e fibre per mesh coarse
vtu_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}_fibers.vtu"
save_vtk_vtu_to_csv(CASE_NAME, geometric_data_dir, coarse_folder_tag, vtu_filename)
save_vtu_arrays_to_csv(CASE_NAME, geometric_data_dir, coarse_folder_tag, vtu_filename)

#export CSV solo nodi e tetra per mesh fine
fine_vtk_filename = f"{CASE_NAME}_{fine_resolution+cdt_fine_um}_fibers.vtk"
save_vtk_vtu_to_csv(CASE_NAME, geometric_data_dir, fine_folder_tag, fine_vtk_filename)
save_vtu_arrays_to_csv(CASE_NAME, geometric_data_dir, fine_folder_tag, fine_vtk_filename)

#export nodi endo rv e lv mesh coarse
input_dir = os.path.join(geometric_data_dir, CASE_NAME)
vtu_path = os.path.join(input_dir, vtu_filename)
output_dir = os.path.join(geometric_data_dir, CASE_NAME, f"{CASE_NAME}_{coarse_folder_tag}")
vtp_filename = f"{CASE_NAME}_{coarse_resolution+cdt_coarse_um}.vtp"
vtp_path = os.path.join(input_dir, vtp_filename)

TAG_LV_ENDO = 3  # Valore per Endocardio LV
TAG_RV_ENDO = 4  # Valore per Endocardio RV
TAG_ARRAY_NAME = 'class'

extract_ids_from_tagged_vtp(
    vtu_path=vtu_path,
    vtp_path=vtp_path,
    tag_array_name=TAG_ARRAY_NAME,
    target_tag_value=TAG_LV_ENDO,
    output_csv=os.path.join(output_dir, CASE_NAME + '_' + coarse_folder_tag + '_boundarynodefield' + '_lvendo' + '.csv')
)
# Genera file RV
extract_ids_from_tagged_vtp(
    vtu_path=vtu_path,
    vtp_path=vtp_path,
    tag_array_name=TAG_ARRAY_NAME,
    target_tag_value=TAG_RV_ENDO,
    output_csv=os.path.join(output_dir, CASE_NAME + '_' + coarse_folder_tag + '_boundarynodefield' + '_rvendo' + '.csv')
)

In [ ]:
torso_case_folder = os.path.join(path_mapping_dict["torso_output_path"], CASE_NAME)
electrodes_vtk_name = CASE_NAME + '_ECG_ELECTRODESmm.vtk'
electrodes_vtk_path = os.path.join(torso_case_folder, electrodes_vtk_name)
output_dir = os.path.join(geometric_data_dir, CASE_NAME)
csv_output = os.path.join(output_dir, CASE_NAME + '_electrode_xyz.csv')
export_electrodes_to_csv(electrodes_vtk_path, csv_output, scale=0.1)  

